# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilalFarooq/Assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Data Contract — Lane 1: Content Refresh

1. ONE ROW MEANS:
   One webpage for one client, measured over 90 days.
   Each row = one unique page at one point in time.

2. TABLE I WILL USE:
   FlyRank/internship-warehouse
   Month slice: 2026-03 (mid-panel month)

3. TIME WINDOW:
   90-day rolling window of search performance.
   Decision made at start of month, outcome observed
   after refresh.

4. WHAT I PREDICT (PROXY):
   trend_direction == 'down' → Label 1 (needs refresh)
   trend_direction != 'down' → Label 0 (okay)

5. WHAT I DELIBERATELY EXCLUDE:
   I exclude the final month (2026-06) from all
   label development — it is the sealed test month.
   I also exclude client names and raw URLs.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [17]:
# QUERY 1: Grain Check — Ek row = kya hai?
print("=== QUERY 1: Grain Check ===")
print("Ek row = one query for one content page for one client")

total_rows = len(df_slice)

# Unique combinations check karo
unique_combos = df_slice.groupby([
    "client_hash_id",
    "content_hash_id",
    "query_hash_id"
]).ngroups

print(f"\nTotal rows: {total_rows}")
print(f"Unique combinations: {unique_combos}")

if total_rows == unique_combos:
    print("✅ Confirmed: one row = one unique query+page+client")
else:
    print(f"Note: Some duplicates exist")
    print(f"Difference: {total_rows - unique_combos}")

=== QUERY 1: Grain Check ===
Ek row = one query for one content page for one client

Total rows: 2414248
Unique combinations: 2414248
✅ Confirmed: one row = one unique query+page+client


In [18]:
# QUERY 2: Row Count aur Date Span
print("=== QUERY 2: Row Count & Date Span ===")

print(f"Month: April 2026")
print(f"Total rows in slice: {len(df_slice)}")
print(f"\nDate span:")
print(f"Window Start: {df_slice['window_start'].min()}")
print(f"Window End:   {df_slice['window_end'].max()}")

print(f"\nUnique clients: {df_slice['client_hash_id'].nunique()}")
print(f"Unique pages:   {df_slice['content_hash_id'].nunique()}")
print(f"Unique queries: {df_slice['query_hash_id'].nunique()}")

=== QUERY 2: Row Count & Date Span ===
Month: April 2026
Total rows in slice: 2414248

Date span:
Window Start: 2026-04-02 00:00:00
Window End:   2026-06-30

Unique clients: 52
Unique pages:   133852
Unique queries: 1180090


In [19]:
# QUERY 3: Availability Check — IS TRUE
print("=== QUERY 3: Availability Check ===")

# Impressions wale rows (available pages)
available = df_slice[df_slice["impressions_90d"] > 0]

print(f"Total rows: {len(df_slice)}")
print(f"Rows with impressions > 0 (available): {len(available)}")
print(f"Availability rate: {len(available)/len(df_slice)*100:.1f}%")
print(f"Rows filtered out: {len(df_slice) - len(available)}")

# Extra check
has_clicks = df_slice[df_slice["clicks_90d"] > 0]
print(f"\nRows with clicks > 0: {len(has_clicks)}")
print(f"Click rate: {len(has_clicks)/len(df_slice)*100:.1f}%")

=== QUERY 3: Availability Check ===
Total rows: 2414248
Rows with impressions > 0 (available): 2414248
Availability rate: 100.0%
Rows filtered out: 0

Rows with clicks > 0: 204152
Click rate: 8.5%


In [10]:
import os, sys, subprocess
from getpass import getpass

# Repo clone
REPO_DIR = "flyrank-ml-internship-starter"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
    "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
    REPO_DIR], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip",
               "install", "-q", "-r", "requirements.txt"], check=True)

# Token maangega — wahan paste karo
token = getpass("HF Token paste karo yahan: ")
os.environ["HF_TOKEN"] = token
print("✅ Setup complete!")

HF Token paste karo yahan: ··········
✅ Setup complete!


In [12]:
from datasets import load_dataset
import pandas as pd

print("Data load ho rahi hai... thoda wait karo!")

# Config name dena zaroor hai!
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",  # ← Yeh add kiya
    token=os.environ["HF_TOKEN"]
)

df = dataset["train"].to_pandas()

# Sirf March 2026 use karo
df_march = df[df["month"] == "2026-03"].copy()

print(f"✅ Data loaded!")
print(f"Total rows (March 2026): {len(df_march)}")
print(f"Columns: {len(df_march.columns)}")
print(f"\nColumn names:")
print(list(df_march.columns))

Data load ho rahi hai... thoda wait karo!


fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

KeyError: 'month'

In [14]:
# window_start se March 2026 filter karo
df["window_start"] = pd.to_datetime(df["window_start"])

# Mid-panel month filter
df_march = df[df["window_start"].dt.to_period("M") == "2026-03"].copy()

print(f"✅ March 2026 slice ready!")
print(f"Total rows (March 2026): {len(df_march)}")
print(f"\nDate range:")
print(f"Min: {df_march['window_start'].min()}")
print(f"Max: {df_march['window_start'].max()}")

✅ March 2026 slice ready!
Total rows (March 2026): 0

Date range:
Min: NaT
Max: NaT


In [15]:
# Kaunsi dates available hain?
print("=== Available Date Ranges ===")
print(df["window_start"].unique()[:10])

print(f"\nMin date: {df['window_start'].min()}")
print(f"Max date: {df['window_start'].max()}")

# Unique months
df["window_start"] = pd.to_datetime(df["window_start"])
df["month"] = df["window_start"].dt.to_period("M")
print(f"\n=== Available Months ===")
print(df["month"].value_counts().sort_index())

=== Available Date Ranges ===
<DatetimeArray>
['2026-04-02 00:00:00']
Length: 1, dtype: datetime64[ns]

Min date: 2026-04-02 00:00:00
Max date: 2026-04-02 00:00:00

=== Available Months ===
month
2026-04    2414248
Freq: M, Name: count, dtype: int64


In [16]:
# April 2026 use karo (yahi available hai)
df_slice = df[df["month"] == "2026-04"].copy()

print(f"✅ April 2026 slice ready!")
print(f"Total rows: {len(df_slice)}")
print(f"\nDate range:")
print(f"Min: {df_slice['window_start'].min()}")
print(f"Max: {df_slice['window_start'].max()}")
print(f"\nColumns: {list(df_slice.columns)}")

✅ April 2026 slice ready!
Total rows: 2414248

Date range:
Min: 2026-04-02 00:00:00
Max: 2026-04-02 00:00:00

Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share', 'month']


In [13]:
# Pehle sab columns dekho
print("=== All Columns ===")
print(list(df.columns))
print(f"\nTotal rows: {len(df)}")
print(f"\nFirst 2 rows:")
print(df.head(2).to_string())

=== All Columns ===
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

Total rows: 2414248

First 2 rows:
            client_hash_id           content_hash_id           query_hash_id  query_char_count  query_token_count window_start  window_end  impressions_90d  clicks_90d  impressions_last30  clicks_last30  impressions_prev30  clicks_prev30  avg_position_90d  avg_position_last30  avg_position_prev30  content_total_impressions_90d  content_visible_query_count  rare_query_count  rare_impressions_share  anonymized_impressions_share
0  client_08a6a72ff48e62c0  content_447894f2faf0d2bc  query_58b1b001f

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [21]:
# 5 FEATURES BANANA
print("=== 5 Feature Frame ===")

# Features select karo
features = df_slice[[
    "avg_position_90d",
    "impressions_90d",
    "clicks_90d",
    "avg_position_last30",
    "query_token_count"
]].copy()

print(features.head(5).to_string())
print(f"\nShape: {features.shape}")
print(f"\n=== Missing Values ===")
print(features.isnull().sum())
print(f"\n=== Basic Stats ===")
print(features.describe().round(2))

=== 5 Feature Frame ===
   avg_position_90d  impressions_90d  clicks_90d  avg_position_last30  query_token_count
0         10.818182               11           0                  NaN                  3
1          1.769231               13           0                  NaN                  7
2         23.562500               16           0            24.272727                  2
3          2.200000               55           0            13.000000                  4
4          3.428571               14           0                  NaN                  3

Shape: (2414248, 5)

=== Missing Values ===
avg_position_90d            0
impressions_90d             0
clicks_90d                  0
avg_position_last30    530758
query_token_count           0
dtype: int64

=== Basic Stats ===
       avg_position_90d  impressions_90d  clicks_90d  avg_position_last30  \
count        2414248.00       2414248.00  2414248.00           1883490.00   
mean              19.77            87.71        0.19       

In [22]:
# THE LEAKAGE TRAP
print("=== LEAKAGE EXPERIMENT ===")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import numpy as np

df_model = df_slice.copy()

# Target banana — clicks > 0 matlab page perform kar raha hai
df_model["target"] = (df_model["clicks_90d"] > 0).astype(int)

# LEAKED feature — target se directly derived!
df_model["has_clicks"] = (df_model["clicks_90d"] > 0).astype(int)

# WITH leak
X_leak = df_model[[
    "avg_position_90d",
    "impressions_90d",
    "has_clicks"  # ← LEAKED!
]].fillna(0)

y = df_model["target"]

rf = RandomForestClassifier(n_estimators=5, random_state=42)
rf.fit(X_leak, y)
pred_leak = rf.predict(X_leak)
print(f"Score WITH leaked feature: {precision_score(y, pred_leak):.3f}")
print("☝️ Near perfect — SUSPICIOUS!")

# WITHOUT leak
X_honest = df_model[[
    "avg_position_90d",
    "impressions_90d",
    "query_token_count"
]].fillna(0)

rf2 = RandomForestClassifier(n_estimators=5, random_state=42)
rf2.fit(X_honest, y)
pred_honest = rf2.predict(X_honest)
print(f"\nScore WITHOUT leaked feature: {precision_score(y, pred_honest):.3f}")
print("✅ More realistic — honest number!")
print("\nLesson: Label-derived features = data leakage!")
print("Always ask: knowable BEFORE the label?")

=== LEAKAGE EXPERIMENT ===
Score WITH leaked feature: 1.000
☝️ Near perfect — SUSPICIOUS!

Score WITHOUT leaked feature: 0.890
✅ More realistic — honest number!

Lesson: Label-derived features = data leakage!
Always ask: knowable BEFORE the label?


## 5 Features — Available When?

1. avg_position_90d → Knowable at decision moment because
   it is the average search position over past 90 days,
   measured BEFORE any refresh decision is made.

2. impressions_90d → Knowable at decision moment because
   it is historical search impressions already collected
   over past 90 days.

3. clicks_90d → Knowable at decision moment because
   it reflects actual user clicks in past 90 days,
   a past fact not a future one.

4. avg_position_last30 → Knowable at decision moment because
   it shows recent 30-day trend in position,
   already observed before decision.
   Note: 530,758 missing values — needs imputation.

5. query_token_count → Knowable at decision moment because
   it is a property of the search query itself,

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## One Named Limitation

This slice covers only 52 clients and one time window
(April 2026).

The proxy label (clicks > 0) assumes that a page with
zero clicks needs improvement — but this may not be true:

- Some queries are informational (no click needed)
- Some pages rank well but get zero clicks naturally
- Position alone does not explain click behavior

This is a limitation of the proxy label, not the model.
Results are directional and decision-support only —
not causal proof.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self Check ✅

- ✅ Contract: 5 answers written
- ✅ Query 1: Grain confirmed (one row = one query+page+client)
- ✅ Query 2: Row count & date span shown
- ✅ Query 3: Availability checked (100%)
- ✅ 5 features built with availability explained
- ✅ Leakage trap shown and removed
- ✅ One limitation named
- ✅ No client names or private data
- ✅ Careful words used: proxy, directional, observed